# 01 · PDF, printed — the baseline path

**In → out:** a born-digital PDF page → a `DoclingDocument` (structured
text, tables, reading order) → JSON + Markdown + a human-readable element
tree.

This notebook keeps the actual conversion path minimal: OCR-engine
resolution, accelerator selection, and building the `DocumentConverter`.
Chunking is the next stage's job (`02-chunk`), not this one.

**First run is slow.** Building the converter loads docling's layout and
table-structure models — expect ~30-60s the first time in a session, not
on every cell. No API key is needed for this notebook.

## What this notebook defines and demonstrates

| Name | What it does | Example |
|---|---|---|
| `build_pdf_converter` | Builds a docling `DocumentConverter` restricted to the PDF pipeline, with OCR engine/accelerator/table-mode resolved | `build_pdf_converter(repo_root=REPO_ROOT, ocr=False)` |
| `_resolve_ocr_options` | Resolves an OCR engine name into the concrete `OcrOptions` docling expects, or raises if OCR flags are passed with `ocr=False` | called internally by `build_pdf_converter` |
| `_resolve_accelerator_options` | Resolves a device string (`"auto"`/`"cpu"`/`"cuda"`/`"mps"`) into docling's `AcceleratorOptions` | called internally by `build_pdf_converter` |
| `_ensure_model_cache` | Points docling's Hugging Face model cache at a repo-local directory instead of the user's home dir | called internally by `build_pdf_converter` |
| `extract_pdf` | Converts a `.pdf` file into JSON, Markdown, and an element-tree run artifact | `extract_pdf(SAMPLE_PDF, RUN_DIR, ocr=False)` |
| `_pdf_metadata` | Reads a PDF's own metadata (title, page count, ...) via PyMuPDF | called internally by `extract_pdf` |
| `_export_element_tree` | Captures `document.print_element_tree()` output as a string for the run artifact | called internally by `extract_pdf` |

## Step 1 — locate the repo root and confirm the environment

Before anything else, resolve `REPO_ROOT` (the kernel's cwd is this notebook's own directory, not the repo root) and print which API keys are present, so the offline/no-key path this notebook takes is stated up front rather than discovered by a later failure.

In [ ]:
import sys
from pathlib import Path

# The kernel's cwd is this notebook's own directory (that's how Jupyter
# starts kernels), not the repo root -- so a bare `import nbio` fails two
# directories down unless the repo root goes on sys.path first. Same
# walk-up nbio.py's own bootstrap() uses internally.
_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from __future__ import annotations

import os
from pathlib import Path

# Quiets a wall of "Could not initialize NNPACK" warnings that some
# torch-backed OCR/layout models print on older CPUs with no AVX2/FMA —
# harmless, but it drowns the actual output on a notebook this short.
os.environ.setdefault("TORCH_CPP_LOG_LEVEL", "ERROR")

import nbio

REPO_ROOT = nbio.bootstrap()
nbio.show_environment()

## Step 2 — shared OCR/accelerator types and the HF-cache helper

`_resolve_ocr_options`, `_resolve_accelerator_options` and `build_pdf_converter` build the `DocumentConverter`. OCR stays resolvable (not deleted) because a page that *looks* clean can still be a scanned image with no text layer — the flag exists so that case doesn't fail silently. This first cell sets up the shared types and the repo-local model-cache helper the rest of Step 2-5 build on.

In [ ]:
from typing import Any, Literal

OcrEngine = Literal["auto", "easyocr", "rapidocr"]
Device = Literal["auto", "cpu", "cuda", "mps"]
TableMode = Literal["accurate", "fast"]

# RapidOCR (this docling version) only ships model bundles for these language
# families, selected by name rather than ISO code.
_RAPIDOCR_LANG_FAMILIES = {"english", "chinese", "latin"}


def _ensure_model_cache(repo_root: Path) -> None:
    """Use a repo-local HF cache when the user hasn't set one, so a fresh
    clone doesn't silently write model weights into the user's home dir."""
    if os.environ.get("HF_HOME") or os.environ.get("HUGGINGFACE_HUB_CACHE"):
        return
    cache_dir = repo_root / ".cache" / "huggingface"
    cache_dir.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(cache_dir)

## Step 3 — resolve the OCR engine

`_resolve_ocr_options` turns `ocr` / `ocr_engine` / `ocr_full_page` / `ocr_lang` into the concrete `OcrOptions` docling expects, and raises if OCR flags are passed while `ocr=False` — a page that *looks* clean can still be a scanned image with no text layer, so this flag exists to keep that case from failing silently.

In [ ]:
def _resolve_ocr_options(
    *, ocr: bool, ocr_engine: OcrEngine, ocr_full_page: bool, ocr_lang: list[str] | None
):
    """Return (resolved_engine, OcrOptions | None).

    Raises if full-page mode, an explicit engine, or a language hint is
    passed without ``ocr=True``: silently ignoring them would be the exact
    "partial transcription, no error" failure this guards against.
    """
    if not ocr:
        if ocr_engine != "auto" or ocr_full_page or ocr_lang:
            raise ValueError(
                "ocr_engine / ocr_full_page / ocr_lang have no effect "
                "without ocr=True (OCR is off by default)."
            )
        return ocr_engine, None

    from docling.datamodel.pipeline_options import EasyOcrOptions, OcrAutoOptions, RapidOcrOptions

    resolved_engine = ocr_engine
    if resolved_engine == "auto":
        return resolved_engine, OcrAutoOptions(force_full_page_ocr=ocr_full_page)
    if resolved_engine == "easyocr":
        kwargs: dict[str, Any] = {"force_full_page_ocr": ocr_full_page}
        if ocr_lang:
            kwargs["lang"] = ocr_lang
        return resolved_engine, EasyOcrOptions(**kwargs)
    if resolved_engine == "rapidocr":
        langs = ocr_lang or RapidOcrOptions.model_fields["lang"].default
        bad = [l for l in langs if l not in _RAPIDOCR_LANG_FAMILIES]
        if bad:
            raise ValueError(
                f"rapidocr only ships {sorted(_RAPIDOCR_LANG_FAMILIES)} language "
                f"families in this docling version, got {bad!r}."
            )
        return resolved_engine, RapidOcrOptions(lang=langs, force_full_page_ocr=ocr_full_page)
    raise ValueError(f"Unknown ocr_engine: {ocr_engine!r}")

## Step 4 — resolve the accelerator options

`_resolve_accelerator_options` turns a device string (`auto`/`cpu`/`cuda`/`mps`) plus an optional thread count into docling's `AcceleratorOptions`.

In [ ]:
def _resolve_accelerator_options(*, device: Device, threads: int | None):
    from docling.datamodel.pipeline_options import AcceleratorDevice, AcceleratorOptions

    device_enum = {
        "auto": AcceleratorDevice.AUTO,
        "cpu": AcceleratorDevice.CPU,
        "cuda": AcceleratorDevice.CUDA,
        "mps": AcceleratorDevice.MPS,
    }[device]
    kwargs: dict[str, Any] = {"device": device_enum}
    if threads is not None:
        kwargs["num_threads"] = threads
    return AcceleratorOptions(**kwargs)

## Step 5 — assemble the `DocumentConverter`

`build_pdf_converter` wires the OCR options, accelerator options, table-structure mode, and the PDF-only `allowed_formats` gate together into one docling `DocumentConverter`.

In [ ]:
def build_pdf_converter(
    *,
    repo_root: Path,
    ocr: bool = False,
    ocr_engine: OcrEngine = "auto",
    ocr_full_page: bool = False,
    ocr_lang: list[str] | None = None,
    device: Device = "auto",
    threads: int | None = None,
    table_mode: TableMode = "accurate",
    images_scale: float = 2.0,
):
    """Build a docling `DocumentConverter` restricted to the PDF pipeline.
    Returns `(converter, resolved_ocr_engine)`.
    """
    resolved_ocr_engine, ocr_options = _resolve_ocr_options(
        ocr=ocr, ocr_engine=ocr_engine, ocr_full_page=ocr_full_page, ocr_lang=ocr_lang
    )

    _ensure_model_cache(repo_root)
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import (
        PdfPipelineOptions,
        TableFormerMode,
        TableStructureOptions,
    )
    from docling.document_converter import DocumentConverter, PdfFormatOption

    pipeline_options = PdfPipelineOptions(
        do_ocr=ocr,
        do_table_structure=True,
        generate_page_images=True,
        generate_picture_images=True,
        images_scale=images_scale,
    )
    pipeline_options.table_structure_options = TableStructureOptions(
        do_cell_matching=True,
        mode=TableFormerMode.ACCURATE if table_mode == "accurate" else TableFormerMode.FAST,
    )
    pipeline_options.accelerator_options = _resolve_accelerator_options(device=device, threads=threads)
    if ocr_options is not None:
        pipeline_options.ocr_options = ocr_options

    converter = DocumentConverter(
        # `allowed_formats` is the actual PDF-only gate on docling's side —
        # without it, docling still assigns a default pipeline to any image
        # you hand it (see the note below on the PDF-only constraint).
        allowed_formats=[InputFormat.PDF],
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)},
    )
    return converter, resolved_ocr_engine

## Step 6 — build the converter once and look at it

Before `extract_pdf` is allowed to depend on `build_pdf_converter`, build one converter with the defaults and print the resolved OCR engine — real output, not an assumption, before moving on.

In [ ]:
# A first look at the object this whole notebook exists to build, before
# anything downstream (extract_pdf) is allowed to depend on it.
demo_converter, demo_engine = build_pdf_converter(repo_root=REPO_ROOT, ocr=False)
print("converter built OK, resolved OCR engine:", demo_engine)
print(type(demo_converter))

## Extracting the sample page

`sample-data/printed-page.pdf` is a synthetic, born-digital PDF built for
this repo (title, two short original paragraphs, one ruled table) — see the
stage `README.md` for why it's synthetic rather than a real scanned page.
Because it's born-digital (it carries a real text layer), OCR stays off:
that is the "clean printed PDF" case this notebook exists to cover.

## Step 7 — read a PDF's own metadata

`_pdf_metadata` opens the PDF with PyMuPDF (`fitz`) and returns its embedded metadata plus page count — one small piece of the run artifact `extract_pdf` will assemble next.

In [ ]:
import contextlib
import io
import json
import time


def _pdf_metadata(pdf_path: Path) -> dict:
    import fitz

    doc = fitz.open(str(pdf_path))
    meta = dict(doc.metadata)
    meta["page_count"] = doc.page_count
    doc.close()
    return meta

## Step 8 — capture the element tree as text

`_export_element_tree` redirects `document.print_element_tree()` (which only prints) into a string, so it can be written to the run artifact alongside the JSON and Markdown exports.

In [ ]:
def _export_element_tree(document) -> str:
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        document.print_element_tree()
    return buffer.getvalue()

## Step 9 — the `extract_pdf` function this notebook exists to show

`extract_pdf` ties `build_pdf_converter`, `_pdf_metadata`, and `_export_element_tree` together: convert the PDF, save JSON + Markdown + element tree, and return a manifest `dict` describing the run.

In [ ]:
def extract_pdf(pdf_path: Path, out_dir: Path, *, ocr: bool = False) -> dict:
    """Converts a PDF into structured JSON + Markdown + an element tree,
    trimmed to this stage's own concern. The manifest below is a plain
    dict written as a run artifact via `nbio` rather than a dedicated
    dataclass — that level of bookkeeping belongs with the links/chunking
    sidecars a later stage owns, not here.
    """
    pdf_path = pdf_path.expanduser().resolve()
    if not pdf_path.is_file():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")
    if pdf_path.suffix.lower() != ".pdf":
        # This is a deliberate suffix check, not a docling limitation.
        # Docling itself will build a default pipeline for an image if you
        # hand it one; this function simply never offers it the chance to.
        raise ValueError(f"Expected a .pdf file, got: {pdf_path}")

    out_dir.mkdir(parents=True, exist_ok=True)
    stem = pdf_path.stem

    converter, resolved_engine = build_pdf_converter(repo_root=REPO_ROOT, ocr=ocr)

    started = time.perf_counter()
    result = converter.convert(str(pdf_path))
    seconds = round(time.perf_counter() - started, 2)
    document = result.document

    docling_json = out_dir / f"{stem}.docling.json"
    document.save_as_json(docling_json, indent=2)

    markdown_md = out_dir / f"{stem}.md"
    document.save_as_markdown(markdown_md)

    (out_dir / f"{stem}.elements.txt").write_text(_export_element_tree(document), encoding="utf-8")

    counts = {
        "texts": len(document.texts),
        "tables": len(document.tables),
        "pictures": len(document.pictures),
        "pages": len(document.pages) if hasattr(document, "pages") else 0,
    }

    record = {
        "source_pdf": str(pdf_path),
        "ocr": ocr,
        "ocr_engine": resolved_engine,
        "counts": counts,
        "pdf_metadata": _pdf_metadata(pdf_path),
        "conversion_seconds": seconds,
        "docling_json": str(docling_json),
        "markdown_md": str(markdown_md),
    }
    (out_dir / f"{stem}.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
    return record

## Step 10 — run it on the anchor sample and look at the result

`sample-data/printed-page.pdf` is a synthetic, born-digital PDF built for this repo (title, two short original paragraphs, one ruled table). Because it's born-digital (it carries a real text layer), OCR stays off — that is the "clean printed PDF" case this notebook exists to cover. Run `extract_pdf` on it and print the real manifest it returns.

In [ ]:
SAMPLE_PDF = Path("sample-data/printed-page.pdf").resolve()
RUN_DIR = REPO_ROOT / "runs" / "01-extract-demo" / "extract"

record = extract_pdf(SAMPLE_PDF, RUN_DIR, ocr=False)
nbio.show_json(record)

## Step 11 — read the same record back from disk

The same record, read back the way a later stage would — via `nbio`, from the run artifact on disk, not the Python variable still in memory.

In [ ]:
# The same record, read back the way a later stage would — via nbio, from
# the run artifact on disk, not the Python variable still in memory.
page = nbio.page("01-extract-demo", "extract", "printed-page")
print("conversion_seconds:", page["conversion_seconds"])
print("counts:", page["counts"])
print()
print(Path(page["markdown_md"]).read_text(encoding="utf-8")[:500])

## The PDF-only constraint, precisely

`extract_pdf` above refuses anything whose suffix isn't `.pdf` — that's an
application-level check, not a technical limit of docling. The cell below
proves the distinction: docling itself will happily build a default
pipeline for `sample-data/handwriting-sample.png` if you don't restrict
`allowed_formats`.

The reason this stage still keeps images on a separate path (`03-
orientation`, `04-handwriting-ocr`) isn't that docling can't read them —
it's a deliberate design choice:

> PDFs go through Docling for structured, born-digital extraction.
> Individual page images (scans, photos, uploads) go through a vision/OCR
> path instead — Docling is not involved.

A contributor who feeds `03-orientation`'s page image into this notebook's
`build_pdf_converter` and finds it "just works" hasn't found a bug in the
PDF-only claim above — they've found the same asymmetry this note names.

## Step 12 — prove the PDF-only gate is a choice, not a docling limit

Build an *unrestricted* `DocumentConverter` (no `allowed_formats` gate) and hand it `sample-data/handwriting-sample.png` directly — no PDF, no `extract_pdf`, no gate. If docling still converts it, the PDF-only check in `extract_pdf` is confirmed to be this notebook's own application-level choice.

In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions

unrestricted = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=PdfPipelineOptions(do_ocr=True))}
)
image_path = Path("sample-data/handwriting-sample.png").resolve()
result = unrestricted.convert(str(image_path))
print("docling converted a raw PNG with no PDF involved at all:")
print(" ", [t.text for t in result.document.texts])

## What this stage covers

| Kept | Left to other stages | Why |
|---|---|---|
| OCR-engine resolution, accelerator options, table-mode | chunking | belongs to `02-chunk` |
| `DocumentConverter` build, JSON/Markdown/element-tree export | hyperlink extraction | out of scope for this stage |
| PDF-suffix gate | — | a plain dict + `nbio` run artifact is enough for this stage's manifest |

See `02-tables-and-layout.ipynb` for what happens to the table this page
carries, and `03-orientation.ipynb` / `04-handwriting-ocr.ipynb` for the
image-only path this notebook deliberately doesn't take.